In [10]:
import tensorflow as tf 
import numpy as np

ds = tf.data.Dataset.range(10) # делаем датасет из чисел от 0 до 9
ds = ds.map(lambda x: x * 2) # применяем функцию к каждому элементу
ds = ds.batch(3) # собираем элементы в батчи по три штуки

for batch in ds.take(2): # берем первые 2 батча
    print(batch.numpy())

[0 2 4]
[ 6  8 10]


In [7]:
ds = tf.data.Dataset.range(10).shuffle(buffer_size=10).batch(5)

for epoch in range(2):
    print(f"Epoch: {epoch + 1}")
    for batch in ds:
        print(batch.numpy())

Epoch: 1
[6 5 8 0 3]
[1 2 9 7 4]
Epoch: 2
[7 9 3 6 5]
[1 4 8 0 2]


In [9]:
images = tf.random.uniform((12, 28, 28, 1), maxval=255, minval=1, dtype=tf.float32)
labels = tf.range(12) % 10

def preprocess(image, label):
    image = tf.image.resize(image, (32, 32)) # меняем размер изображения
    image = image / 255.0 # нормализуем пиксели 
    image = tf.image.random_flip_left_right(image) # случайное отражение
    return image, label

ds = (
    tf.data.Dataset.from_tensor_slices((images, labels))
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE) # позволяю тф самому выбрать колво параллельных операций
    .batch(4)
    .prefetch(tf.data.AUTOTUNE)
)

for x, y in ds.take(1):
    print(x.shape, y.numpy())

(4, 32, 32, 1) [0 1 2 3]


In [ ]:
dataset = (
    tf.data.Dataset.from_tensor_slices((X, y))
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .shuffle(buffer_size=1000)
    .batch(32)
    .prefetch(tf.data.AUTOTUNE)
)

In [11]:
def make_dataset(X, y, batch_size=32, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))

    def preprocess(x, label):
        x = tf.cast(x, tf.float32) / 255.0
        return x, label

    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

    if training:
        ds = ds.cache()
        shuffle_buffer = min(10_000, len(X))
        ds = ds.shuffle(buffer_size=shuffle_buffer, reshuffle_each_iteration=True)
    else:
        ds = ds.cache()

    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds

In [12]:
X = np.random.randint(0, 256, size=(200, 28, 28, 1)).astype(np.float32)
y = np.random.randint(0, 10, size=(200,)).astype(np.int64)

train_ds = make_dataset(X[:160], y[:160], batch_size=16, training=True)
val_ds = make_dataset(X[160:], y[160:], batch_size=16, training=False)

In [13]:
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28, 1)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(10),
])

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
)

/usr/lib/python3.14/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [14]:
history = model.fit(
    train_ds,
    epochs=2,
    validation_data=val_ds,
    verbose=2
)

Epoch 1/2


/usr/lib/python3.14/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


10/10 - 2s - 173ms/step - accuracy: 0.0938 - loss: 2.5094 - val_accuracy: 0.1000 - val_loss: 2.3946
Epoch 2/2
10/10 - 0s - 7ms/step - accuracy: 0.1000 - loss: 2.3248 - val_accuracy: 0.0500 - val_loss: 2.3656


In [17]:
ds = tf.data.Dataset.range(20)
ds = ds.map(lambda x: x  *2)
ds = ds.batch(5)

for batch in ds.take(2):
    print(batch.numpy())

[0 2 4 6 8]
[10 12 14 16 18]
